# 07 · Cross-model geometry — organism matrix · probe transfer · layer CKA

Pure offline linear algebra over the frozen-frame directions from `06b` (probe, desirability, CAA) and
`06c` (Likert stance, open-ended, induced-shift), plus the cached activations `acts_<model>.npz`. **No
model loading** — just numpy. Every direction lives in the shared Qwen3-8B basis at matched layer ids,
so cosines and CKA are meaningful across models and estimators.

Answers the headline questions:
- **Organism matrix** — is `light ≈ −dark`? `happy ≈ −depressed`? Are the dark↔light and
  depressed↔happy axes **orthogonal** (antagonism vs internalizing = distinct directions in the model)?
- **Probe transfer** — apply model A's desirability direction to model B's activations; how much rank
  info survives? (shared direction vs model-specific encoding).
- **Layer CKA** — *find* the corresponding layer between models instead of assuming L21↔L25; quantify
  the fine-tune's representational drift.
- **Convergence** — does the **induced-shift** (what SFT installed) align with the **prompting/CAA** and
  **probe** directions? (your locked experiment).

## 1. Setup (light — numpy/scipy/matplotlib only)

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U numpy scipy matplotlib
DRIVE = mount_drive()
import pathlib, json, pickle, numpy as np
assert DRIVE is not None
OUT = DRIVE / "directions_v1"
assert OUT.exists(), f"{OUT} not found — run 06b/06c first"
MODELS = json.load(open("/content/dt_rl/notebooks/organisms.json"))["models"]
NAMES = [m["name"] for m in MODELS]
print("organisms:", NAMES, "| dir:", OUT)

## 2. Loaders + helpers

In [ ]:
import numpy as np, pickle, pathlib
def _unit(v):
    v = np.asarray(v, np.float32); n = np.linalg.norm(v); return v/n if n else v

def load_bundle(name, kind):
    # kind in {desirability, clinical, pc, train, trainopen, shift}
    p = OUT/f"control_vectors_{kind}_{name}.pkl"
    return pickle.load(open(p,"rb")) if p.exists() else None

def bundle_dir(name, kind, trait, L):
    b = load_bundle(name, kind)
    if b is None: return None
    d = b["vectors"].get(trait)
    return None if d is None else d.get(int(L))

def probe_dir(name, L):
    p = OUT/f"probe_{name}_all.npz"
    if not p.exists(): return None
    z = np.load(p); ls = list(z["layers"])
    return z["unit"][ls.index(int(L))] if int(L) in ls else None

def load_acts(name):
    p = OUT/f"acts_{name}.npz"
    if not p.exists(): return None
    z = np.load(p)
    return {"X": z["X"].astype(np.float32), "layers": list(z["layers"]),
            "task_ids": list(z["task_ids"]), "mu": z["mu"]}

def acts_at(A, L):
    return A["X"][:, A["layers"].index(int(L)), :]   # [n, d] at layer L

def band_layers():
    for n in NAMES:
        p = OUT/f"probe_{n}_all.npz"
        if p.exists(): return list(np.load(p)["layers"])
        a = load_acts(n)
        if a: return a["layers"]
    raise FileNotFoundError("no probe/acts npz found")

LAYERS = band_layers(); LMID = LAYERS[len(LAYERS)//2]
print(f"band {LAYERS[0]}..{LAYERS[-1]} ({len(LAYERS)} layers); default L={LMID}")

def cos_matrix(vecs):
    labs=[k for k,v in vecs.items() if v is not None]; V=[_unit(vecs[k]) for k in labs]
    M=np.array([[float(a@b) for b in V] for a in V]); return labs, M

def show(labs, M, title, fmt="{:+.2f}"):
    print(f"\n### {title} ###")
    w=max(len(x) for x in labs)+1
    print(" "*w + " ".join(x[:7].rjust(7) for x in labs))
    for i,a in enumerate(labs):
        print(a.ljust(w) + " ".join(fmt.format(M[i,j]).rjust(7) for j in range(len(labs))))

def heatmap(labs_r, labs_c, M, title, vmin=-1, vmax=1, cmap="coolwarm"):
    try:
        import matplotlib.pyplot as plt
    except Exception: return
    fig,ax=plt.subplots(figsize=(1+0.6*len(labs_c), 1+0.6*len(labs_r)))
    im=ax.imshow(M, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(labs_c))); ax.set_xticklabels(labs_c, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labs_r))); ax.set_yticklabels(labs_r, fontsize=8)
    for i in range(len(labs_r)):
        for j in range(len(labs_c)):
            ax.text(j,i,f"{M[i,j]:+.2f}",ha="center",va="center",fontsize=7,
                    color="white" if abs(M[i,j])>0.6 else "black")
    ax.set_title(title, fontsize=10); fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()
print("helpers ready")

## 3. Organism matrix — induced-shift geometry
The `#2` induced-shift (what SFT installed) for each organism, cosine across models. Look for
`light≈−dark`, `happy≈−depressed` (off-diagonal ≈ −1) and dark⊥depressed (≈ 0).

In [ ]:
shift = {n: bundle_dir(n, "shift", "induced_shift", LMID) for n in NAMES if n != "base"}
shift = {k:v for k,v in shift.items() if v is not None}
if len(shift) >= 2:
    labs, M = cos_matrix(shift); show(labs, M, f"induced-shift cosine @ L{LMID}"); heatmap(labs, labs, M, f"organism induced-shift @ L{LMID}")
    for a in labs:
        for b in labs:
            if a<b:
                c=float(_unit(shift[a])@_unit(shift[b]))
                tag = "OPPOSITE axis" if c<-0.5 else ("~orthogonal" if abs(c)<0.3 else ("aligned" if c>0.5 else ""))
                if tag: print(f"  {a} vs {b}: cos={c:+.2f}  {tag}")
else:
    print("need >=2 non-base organisms with a shift vector; have:", list(shift))

## 4. Trait-axis rotation across models
For one trait direction (default the Likert `dark` stance, `#1`), how aligned is it across organisms?
Rotation away from 1.0 = the fine-tune moved that trait's axis. Swap `TRAIT`/`KIND` freely
(`kind="clinical"` + a mechanism, `kind="train"` + `"depression"`, etc.).

In [ ]:
TRAIT, KIND = "dark", "train"
tv = {n: bundle_dir(n, KIND, TRAIT, LMID) for n in NAMES}
tv = {k:v for k,v in tv.items() if v is not None}
if len(tv) >= 2:
    labs, M = cos_matrix(tv); show(labs, M, f"{KIND}:{TRAIT} direction cosine across models @ L{LMID}")
    heatmap(labs, labs, M, f"{KIND}:{TRAIT} across models @ L{LMID}")
else:
    print(f"{KIND}:{TRAIT} present in <2 models:", list(tv))

## 5. Cross-estimator convergence (within model)
Do the different ways of naming the same trait point the same way? For each model, cosine among:
induced-shift (#2), Likert-dark (#1), desirability probe (04-style), desirability CAA (05-style). The
key science: does **what SFT installed** align with **what prompting elicits**?

In [ ]:
for n in NAMES:
    est = {"shift":     bundle_dir(n,"shift","induced_shift",LMID),
           "likert_dark":bundle_dir(n,"train","dark",LMID),
           "desir_CAA": bundle_dir(n,"desirability","desirability",LMID),
           "probe":     probe_dir(n, LMID)}
    est = {k:v for k,v in est.items() if v is not None}
    if len(est) >= 2:
        labs, M = cos_matrix(est); show(labs, np.abs(M), f"{n}: |cos| among estimators @ L{LMID}")

## 6. Probe transfer matrix
Project model **B**'s activations onto model **A**'s desirability probe direction, score how well that
ranks B's tasks by B's own μ (pairwise accuracy — threshold/scale-free, so no refit). Diagonal =
within-model; high off-diagonal = the desirability direction is shared, not model-specific.

In [ ]:
def pairwise_acc(pred, true):
    dp=np.sign(pred[:,None]-pred[None,:]); dt=np.sign(true[:,None]-true[None,:])
    m=np.triu(np.ones_like(dp,bool),1); return float((dp[m]==dt[m]).mean())

acts = {n:load_acts(n) for n in NAMES}; acts={k:v for k,v in acts.items() if v is not None}
avail=[n for n in NAMES if probe_dir(n,LMID) is not None and n in acts]
if len(avail)>=1:
    # align tasks across all available models
    common=set(acts[avail[0]]["task_ids"])
    for n in avail: common &= set(acts[n]["task_ids"])
    common=sorted(common)
    def aligned(n):
        idx=[acts[n]["task_ids"].index(t) for t in common]
        return acts_at(acts[n],LMID)[idx], acts[n]["mu"][idx]
    T=np.zeros((len(avail),len(avail)))
    for i,A in enumerate(avail):
        uA=_unit(probe_dir(A,LMID))
        for j,B in enumerate(avail):
            XB,muB=aligned(B); T[i,j]=pairwise_acc(XB@uA, muB)
    show(avail, T, f"probe transfer pairwise-acc @ L{LMID} (row=probe A -> col=acts B)", "{:.2f}")
    heatmap(avail, avail, T, f"probe transfer @ L{LMID}", vmin=0.5, vmax=1.0, cmap="viridis")
    print(f"({len(common)} shared tasks)")
else:
    print("need probe + acts for >=1 model")

## 7. Per-layer CKA — find the corresponding layer
Linear CKA between every band-layer of model A and model B on the cached activations. The argmax per
row is A-layer→B-layer correspondence: if it bends off the diagonal, the fine-tune moved trait
representations to a different depth (your L21→L25 drift, measured).

In [ ]:
def cka(X, Y):
    X=X-X.mean(0); Y=Y-Y.mean(0)
    hsic=np.linalg.norm(Y.T@X)**2
    return float(hsic/((np.linalg.norm(X.T@X)*np.linalg.norm(Y.T@Y)) or 1.0))

def cka_matrix(A, B):
    common=sorted(set(acts[A]["task_ids"])&set(acts[B]["task_ids"]))
    ia=[acts[A]["task_ids"].index(t) for t in common]; ib=[acts[B]["task_ids"].index(t) for t in common]
    Ls=acts[A]["layers"]; M=np.zeros((len(Ls),len(Ls)))
    for r,La in enumerate(Ls):
        Xa=acts[A]["X"][ia, Ls.index(La), :]
        for c,Lb in enumerate(Ls):
            M[r,c]=cka(Xa, acts[B]["X"][ib, Ls.index(Lb), :])
    return Ls, M

PAIR = ("base", "dark")   # <- change to any two models with acts
if all(p in acts for p in PAIR):
    Ls, M = cka_matrix(*PAIR)
    heatmap([str(l) for l in Ls], [str(l) for l in Ls], M, f"CKA {PAIR[0]}(rows) vs {PAIR[1]}(cols)", vmin=0, vmax=1, cmap="viridis")
    drift=[Ls[int(np.argmax(M[r]))]-Ls[r] for r in range(len(Ls))]
    print(f"{PAIR[0]}->{PAIR[1]} best-match layer offset (median): {int(np.median(drift)):+d} "
          f"(range {min(drift):+d}..{max(drift):+d})")
else:
    print("need acts for both of", PAIR, "; have:", list(acts))

## Summary
Everything here reads the frozen-frame bundles — re-run any cell with a different `L`, `TRAIT/KIND`, or
`PAIR` without touching a GPU. The four panels together test the core claim: **is psychopathology
(internalizing) geometrically distinct from antagonism inside the model, and does what the fine-tune
installed match what prompting elicits?** Organism matrix (§3) + trait rotation (§4) give the
distinctness; convergence (§5) + transfer (§6) give the shared-vs-specific split; CKA (§7) grounds it
all at the right layer.